In [15]:
from openai import OpenAI
import json
import os
import re
from datetime import date, timedelta

import requests
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field

_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

client = OpenAI()

In [16]:
SYSTEM_PROMPT = """You are a friendly, practical AI travel advisor.
Help the user plan a personalized, realistic trip through conversation.
Before generating an itinerary, make sure you know all three essentials:
(1) destination,
(2) trip duration (number of days or dates), and
(3) at least one travel interest (for example food, museums, hiking, beaches,
nightlife, or shopping).

Ask one concise, natural follow-up covering only the missing essentials. Do not
ask for optional preferences if the user has not volunteered them; use sensible
assumptions instead.

If the destination, dates, duration, or other details conflict or are materially
ambiguous, ask the user to clarify before planning. Do not silently choose between
contradictory details. Treat an explicit "surprise me" destination as permission
to suggest a destination and proceed, while briefly stating your choice.

Once the essentials are clear and there is no unresolved contradiction, provide
the complete plan without asking for confirmation. Include a memorable trip title,
a short summary, a day-by-day itinerary with a sensible pace, practical travel
tips, and an estimated per-person budget with currency and a clear breakdown.
State assumptions, especially when dates, departure city, or budget are unknown.

Use the weather and exchange-rate function tools when current weather or currency
conversion is relevant to the user's request or itinerary. Weather forecasts are
only meaningful within the available forecast period; beyond that, describe
seasonal expectations instead. Label retrieved data with its date, and treat
exchange rates as approximate mid-market rates, not guaranteed card or bank rates.
Use web search for current public-transport options, operator information, or
service changes. Prefer official sources. Do not claim live availability or
opening hours. Include clickable source URLs for retrieved facts and recommend
checking details before booking.

Respect any accessibility, dietary, safety, or budget constraints the user mentions.
Be transparent when a specific recommendation or cost is uncertain.
"""

Token totals are accumulated from each API response's usage data. The cost is an estimate using the Standard `gpt-6-luna` rates listed in the [OpenAI API pricing guide](https://developers.openai.com/api/docs/pricing): $0.10 per 1M input tokens, $0.01 per 1M cached input tokens, and $0.50 per 1M output tokens. Update the rates if you change the model or pricing tier. Separate built-in web-search charges are not included.

In [17]:
MODEL = "gpt-6-luna"

PRICING_USD_PER_MILLION = {
    "gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50}
}
history = [{"role": "system", "content": SYSTEM_PROMPT}]

conversation_usage = {
    "input_tokens": 0,
    "cached_input_tokens": 0,
    "output_tokens": 0,
    "estimated_cost_usd": 0.0,
}

class Structure(BaseModel):
    title: str = Field(description="A title for the trip")
    summary: str = Field(description="Trip summary")
    itinerary: str = Field(description="Day-by-day itinerary")
    practical_tips: str = Field(description="Practical travel tips")
    budget: str = Field(description="Estimated budget")


TOOLS = [
    {
        "type": "function",
        "name": "get_weather_forecast",
        "description": "Get a daily weather forecast for a location and dates within the next 16 days.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City and country"},
                "start_date": {"type": "string", "description": "First forecast date in YYYY-MM-DD format"},
                "days": {"type": "integer", "minimum": 1, "maximum": 16},
            },
            "required": ["location", "start_date", "days"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "get_exchange_rate",
        "description": "Get a recent exchange rate and convert an amount between two currencies.",
        "parameters": {
            "type": "object",
            "properties": {
                "base_currency": {"type": "string", "description": "Three-letter currency code"},
                "quote_currency": {"type": "string", "description": "Three-letter currency code"},
                "amount": {"type": "number", "description": "Amount in the base currency"},
            },
            "required": ["base_currency", "quote_currency", "amount"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {"type": "web_search"},
]


def get_weather_forecast(location, start_date, days):
    forecast_start = date.fromisoformat(start_date)
    forecast_end = forecast_start + timedelta(days=days - 1)
    today = date.today()
    if days < 1 or days > 16:
        return {"error": "Choose between 1 and 16 forecast days."}
    if forecast_start < today or forecast_end > today + timedelta(days=15):
        return {"error": "A forecast is only available for dates within the next 16 days. Use seasonal expectations for later trips."}

    geocoding_response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": location, "count": 1, "language": "en", "format": "json"},
        timeout=20,
    )
    geocoding_response.raise_for_status()
    locations = geocoding_response.json().get("results", [])
    if not locations:
        return {"error": f"Could not find a location matching {location!r}."}

    place = locations[0]
    forecast_response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "daily": "weather_code,temperature_2m_max,temperature_2m_min,precipitation_probability_max",
            "timezone": "auto",
            "start_date": forecast_start.isoformat(),
            "end_date": forecast_end.isoformat(),
        },
        timeout=20,
    )
    forecast_response.raise_for_status()
    forecast_data = forecast_response.json()
    return {
        "location": f"{place['name']}, {place.get('country', '')}",
        "timezone": forecast_data.get("timezone"),
        "daily_forecast": forecast_data.get("daily", {}),
        "retrieved_on": today.isoformat(),
        "sources": [
            {"title": "Open-Meteo geocoding", "url": geocoding_response.url},
            {"title": "Open-Meteo forecast", "url": forecast_response.url},
        ],
    }


def get_exchange_rate(base_currency, quote_currency, amount):
    base_currency = base_currency.upper()
    quote_currency = quote_currency.upper()
    if (
        not re.fullmatch(r"[A-Z]{3}", base_currency)
        or not re.fullmatch(r"[A-Z]{3}", quote_currency)
    ):
        return {"error": "Currency codes must each contain exactly three letters."}

    rate_response = requests.get(
        f"https://api.frankfurter.dev/v2/rate/{base_currency}/{quote_currency}",
        timeout=20,
    )
    rate_response.raise_for_status()
    rate_data = rate_response.json()
    return {
        "base_currency": base_currency,
        "quote_currency": quote_currency,
        "rate": rate_data["rate"],
        "rate_date": rate_data["date"],
        "amount": amount,
        "converted_amount": round(amount * rate_data["rate"], 2),
        "sources": [{"title": "Frankfurter exchange rates", "url": rate_response.url}],
    }


def execute_function_tool(name, arguments):
    try:
        if name == "get_weather_forecast":
            return get_weather_forecast(**arguments)
        if name == "get_exchange_rate":
            return get_exchange_rate(**arguments)
        return {"error": f"Unknown function tool: {name}"}
    except (requests.RequestException, KeyError, ValueError) as error:
        return {"error": f"Tool {name} could not retrieve data: {error}"}


def extract_web_sources(response):
    sources = []
    seen_urls = set()
    for output_item in response.output:
        if getattr(output_item, "type", None) != "message":
            continue
        for content in getattr(output_item, "content", []):
            for annotation in getattr(content, "annotations", []) or []:
                if getattr(annotation, "type", None) == "url_citation":
                    url = getattr(annotation, "url", None)
                    if url and url not in seen_urls:
                        sources.append((getattr(annotation, "title", None) or url, url))
                        seen_urls.add(url)
    return sources


def format_as_markdown(trip: Structure, sources=()):
    markdown = f"""# {trip.title}

## Summary
{trip.summary}

## Day-by-day itinerary
{trip.itinerary}

## Practical tips
{trip.practical_tips}

## Estimated budget
{trip.budget}"""
    if sources:
        source_links = "\n".join(f"- [{title}]({url})" for title, url in sources)
        markdown += f"\n\n## Sources\n{source_links}"
    return markdown


def send_request(input):
    return client.responses.parse(
        model=MODEL,
        input=input,
        text_format=Structure,
        tools=TOOLS,
    )


def chat(user_text):
    input_items = list(history)
    input_items.append({"role": "user", "content": user_text})
    tool_sources = []
    total_input_tokens = 0
    total_cached_input_tokens = 0
    total_output_tokens = 0

    for tool_round in range(6):
        response = send_request(input_items)
        usage = response.usage
        total_input_tokens += usage.input_tokens
        total_cached_input_tokens += getattr(
            getattr(usage, "input_tokens_details", None), "cached_tokens", 0
        ) or 0
        total_output_tokens += usage.output_tokens

        function_calls = [
            item for item in response.output
            if getattr(item, "type", None) == "function_call"
        ]
        if not function_calls:
            break
        if tool_round == 5:
            raise RuntimeError("Tool calling exceeded the maximum of six rounds.")

        input_items.extend(response.output)
        for function_call in function_calls:
            function_arguments = json.loads(function_call.arguments)
            tool_result = execute_function_tool(function_call.name, function_arguments)
            for source in tool_result.get("sources", []):
                tool_sources.append((source["title"], source["url"]))
            input_items.append({
                "type": "function_call_output",
                "call_id": function_call.call_id,
                "output": json.dumps(tool_result),
            })

    answer = response.output_text
    history.append({"role": "user", "content": user_text})
    history.append({"role": "assistant", "content": answer})

    conversation_usage["input_tokens"] += total_input_tokens
    conversation_usage["cached_input_tokens"] += total_cached_input_tokens
    conversation_usage["output_tokens"] += total_output_tokens

    rates = PRICING_USD_PER_MILLION[MODEL]
    uncached_input_tokens = total_input_tokens - total_cached_input_tokens
    conversation_usage["estimated_cost_usd"] += (
        uncached_input_tokens * rates["input"]
        + total_cached_input_tokens * rates["cached_input"]
        + total_output_tokens * rates["output"]
    ) / 1_000_000

    print(
        f"Conversation totals: {conversation_usage['input_tokens']:,} input tokens, "
        f"{conversation_usage['output_tokens']:,} output tokens; "
        f"estimated model cost ${conversation_usage['estimated_cost_usd']:.6f} USD"
    )
    sources = tool_sources + extract_web_sources(response)
    return format_as_markdown(response.output_parsed, sources)

In [18]:
from IPython.display import Markdown, display

trip_start = (date.today() + timedelta(days=2)).isoformat()
display(Markdown(chat(
    f"I want to visit Barcelona for 3 days starting {trip_start}. "
    "I love food and architecture. Include the weather forecast and convert "
    "500 USD to EUR."
)))
display(Markdown(chat(
    "What public transport options should I consider in Barcelona? Search for "
    "current official information."
)))

Conversation totals: 19,083 input tokens, 2,141 output tokens; estimated model cost $0.002517 USD


# Barcelona: Gaudí, Gothic Lanes & Catalan Flavours

## Summary
A three-day food-and-architecture trip from Friday, October 16, through Sunday, October 18, 2026. The forecast is unsettled after Friday, so the plan puts outdoor sightseeing on the clearer day and keeps indoor architecture and food stops flexible. Assumptions: one traveler, three sightseeing days, two hotel nights, and airfare not included.

## Day-by-day itinerary
Day 1 — Friday, October 16: Begin with Park Güell while the forecast looks driest; book an entry slot ahead (official general admission is €18). Have lunch in Gràcia, then visit the Sagrada Família in the afternoon. Finish with a relaxed tapas dinner. The basilica’s official October hours list Friday closing at 7 p.m.; check the date-specific ticket options before booking. ([parkguell.barcelona](https://parkguell.barcelona/index.php/en/planning-your-visit/prices-and-times?utm_source=openai))

Day 2 — Saturday, October 17: Make this the indoor Modernisme day, with pre-booked visits to Casa Batlló and Casa Milà (La Pedrera) on Passeig de Gràcia. Take a leisurely lunch nearby, then spend the afternoon over coffee, pastries, or a guided food tasting. The forecast currently suggests a high chance of thunderstorms, so keep outdoor walks short and flexible.

Day 3 — Sunday, October 18: Explore the Gothic Quarter and El Born at an easy pace, visiting the cathedral area and choosing an indoor museum or food stop if rain is heavy. Enjoy a long Catalan lunch, then finish with a covered or sheltered stroll rather than planning around a long outdoor visit. Check Sunday opening and ticket details for any attraction you choose.

## Practical tips
Weather forecast retrieved October 14, 2026: Friday 16°C–25°C, fog possible and 0% maximum precipitation probability; Saturday 16°C–23°C, thunderstorms indicated and 95% maximum precipitation probability; Sunday 12°C–17°C, rain indicated and 90% maximum precipitation probability. Forecasts can change—bring a compact umbrella and light waterproof layer, and recheck before heading out. [Open-Meteo forecast](https://api.open-meteo.com/v1/forecast?latitude=41.38879&longitude=2.15899&daily=weather_code%2Ctemperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_probability_max&timezone=auto&start_date=2026-10-16&end_date=2026-10-18)

For airport metro travel, TMB says L9 Sud connects both airport terminals with the city; a T-casual pass is not valid for airport metro stations, so choose an eligible airport ticket or pass. ([tmb.cat](https://www.tmb.cat/en/visit-barcelona/public-transport/metro-airport?utm_source=openai)) Book the Sagrada Família and Park Güell through their official ticket channels, and verify schedules before setting out; opening times can change. ([sagradafamilia.org](https://sagradafamilia.org/en/faqs?utm_source=openai))

## Estimated budget
Estimated per person for 3 days / 2 nights, excluding flights: lodging €300–€500; food and drinks €120–€180; sightseeing tickets €75–€120; local and airport transport €20–€40; small extras €25–€50. Estimated total: about €540–€890. These are planning ranges, not live prices; lodging varies especially by booking date and room type. On October 14, 2026, $500 converts to approximately €446.38 at the retrieved approximate mid-market rate of 1 USD = 0.89277 EUR; your card or bank rate may differ. [Exchange-rate source](https://api.frankfurter.dev/v2/rate/USD/EUR)

## Sources
- [Open-Meteo geocoding](https://geocoding-api.open-meteo.com/v1/search?name=Barcelona%2C+Spain&count=1&language=en&format=json)
- [Open-Meteo forecast](https://api.open-meteo.com/v1/forecast?latitude=41.38879&longitude=2.15899&daily=weather_code%2Ctemperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_probability_max&timezone=auto&start_date=2026-10-16&end_date=2026-10-18)
- [Frankfurter exchange rates](https://api.frankfurter.dev/v2/rate/USD/EUR)
- [Opening times | Web oficial Park Güell | Barcelona](https://parkguell.barcelona/index.php/en/planning-your-visit/prices-and-times?utm_source=openai)
- [Accessing Barcelona from the airport by metro | Transports Metropolitans de Barcelona](https://www.tmb.cat/en/visit-barcelona/public-transport/metro-airport?utm_source=openai)
- [FAQs - Sagrada Família - Sagrada Familia](https://sagradafamilia.org/en/faqs?utm_source=openai)

Conversation totals: 64,067 input tokens, 5,067 output tokens; estimated model cost $0.008017 USD


# Getting Around Barcelona (October 16–18, 2026)

## Summary
For your three-day visit, use the metro for longer cross-city trips, buses when you want to see the city above ground, and walk around compact areas such as the Gothic Quarter and El Born. Barcelona’s integrated fare system covers metro, bus, tram, FGC and Rodalies trips within the relevant zones. ([tmb.cat](https://www.tmb.cat/en/visit-barcelona/barcelona-in-public-transport/move-around))

## Day-by-day itinerary
Options to consider:

- **Metro:** Usually the quickest way to cover longer distances, including between many major sightseeing areas. On your visit, metro service is scheduled until **2 a.m. Friday night**, runs continuously from **Saturday at 5 a.m. through Sunday night**, and ends at midnight on Sunday. Check live service information before you travel. ([tmb.cat](https://www.tmb.cat/en/visit-barcelona/barcelona-in-public-transport/move-around))
- **City buses:** Useful for shorter trips and seeing the streets along the way. TMB says its bus network is accessible; bus arrivals and route schedules are available through its tools. ([tmb.cat](https://www.tmb.cat/en/visit-barcelona/barcelona-in-public-transport/move-around))
- **Tram, FGC and Rodalies:** These are also part of the integrated network. They may be handy depending on where you stay or if you go beyond central Barcelona; check the zone needed for your destination. ([tmb.cat](https://www.tmb.cat/en/barcelona-fares-metro-bus/zone-map?utm_source=openai))
- **From the airport:** L9 Sud metro serves both airport terminals and takes about 32 minutes to the city network. The airport metro ticket is **€5.90 each way** in 2026; a T-casual is not valid at the airport metro stops. Other public-transport choices listed by TMB include Aerobús, bus 46, the R2 train and NitBus. The R2 airport branch serves Terminal 2, so check your terminal and current service if considering it. ([tmb.cat](https://www.tmb.cat/en/visit-barcelona/public-transport/metro-airport))
- **Aerobús:** A dedicated airport shuttle is another option; its operator describes a route between the airport and central stops including Plaça Catalunya. It is separate from the regular integrated transit ticketing system. Check the operator’s current route, stops and fare before you go. ([aerobusbarcelona.es](https://aerobusbarcelona.es/en/guia-de-viaje/aerobus-journey/?utm_source=openai))

## Practical tips
**Ticket suggestion:** For a solo visitor making around ten local journeys, consider the **T-casual**: it covers 10 journeys and costs **€13 for Zone 1** under the 2026 fares. It is for one person and does not cover the L9 Sud airport stations. For two or more people sharing journeys, the **T-familiar** offers 8 journeys for **€11.50 in Zone 1**; each traveler’s journey uses a validation. The **Hola Barcelona Travel Card** offers unlimited travel for consecutive time periods and includes airport metro trips, so compare it if you expect frequent rides. ([tmb.cat](https://www.tmb.cat/en/barcelona-fares-metro-bus/t-casual?utm_source=openai))

TMB lists the **T-usual at €22.80 for Zone 1**, for unlimited travel over 30 consecutive days; it is personal and non-transferable, so it may be more pass than you need for three days. Integrated fares allow transfers between modes within the ticket’s rules, so validate when changing vehicles and keep within the allowed zones and time. ([tmb.cat](https://www.tmb.cat/es/tarifas-metro-bus-barcelona/precios-titulos-transporte))

For official route planning, ticket details and service notices, use the [TMB public-transport guide](https://www.tmb.cat/en/visit-barcelona/barcelona-in-public-transport/move-around), [2026 fares](https://www.tmb.cat/en/barcelona-fares-metro-bus), [airport transport guide](https://www.tmb.cat/en/visit-barcelona/public-transport/metro-airport), and [Rodalies R2 Nord information](https://rodalies.gencat.cat/en/sobre-rodalies/linies-i-estacions/servei_rodalia_barcelona/r2n/index.html). Information checked October 14, 2026; recheck close to departure for service changes.

## Estimated budget
Indicative transit costs per person, using official 2026 Zone 1 fares: **€13 for a T-casual** (10 local journeys); add **€11.80** if using L9 Sud both ways with two €5.90 airport tickets, for an example total of **€24.80**. If you choose Aerobús instead, budget for its separate fare and check the operator’s current price. Actual spending depends on your number of rides and airport transfer choice. ([tmb.cat](https://www.tmb.cat/en/barcelona-fares-metro-bus/t-casual?utm_source=openai))

## Sources
- [Get around Barcelona by metro and bus | Transports Metropolitans de Barcelona](https://www.tmb.cat/en/visit-barcelona/barcelona-in-public-transport/move-around)
- [Zone map | Transports Metropolitans de Barcelona](https://www.tmb.cat/en/barcelona-fares-metro-bus/zone-map?utm_source=openai)
- [Accessing Barcelona from the airport by metro | Transports Metropolitans de Barcelona](https://www.tmb.cat/en/visit-barcelona/public-transport/metro-airport)
- [Aerobús journey - Monbus Aerobus](https://aerobusbarcelona.es/en/guia-de-viaje/aerobus-journey/?utm_source=openai)
- [T-casual | Transports Metropolitans de Barcelona](https://www.tmb.cat/en/barcelona-fares-metro-bus/t-casual?utm_source=openai)
- [Tarifas metro y bus Barcelona 2026: precios y títulos | Transports Metropolitans de Barcelona](https://www.tmb.cat/es/tarifas-metro-bus-barcelona/precios-titulos-transporte)